# 44 - Build the scaled corpus (100K / 200K / 300K / ~400K nested tiers)

Combines the existing 98,716-company corpus (which holds all gold-labelled data) with the new 300K random sample (notebook 43), and builds four nested corpus-size tiers for the scaling experiment: does retrieval quality hold up as the haystack grows.

**Why nested, not four independent random draws**: each larger tier is the smaller tier plus more random companies layered on top, using one fixed shuffle. That way going from 100K to 400K only ever adds companies, it never swaps out what was already there, so any change in a metric between tiers is attributable to added noise, not to a different sample.

**Why the existing corpus is in every tier**: all current gold labels live there. If a tier's random subset happened to drop some of those companies, Recall would degrade for a reason that has nothing to do with retrieval quality.

This notebook only builds the combined text data and the tier membership flags, it does not encode anything. Notebooks 45/46/47 do the actual encoding (MiniLM, GTE-large, Linq-Embed-Mistral respectively), each encoding this same combined pool once, so no re-encoding happens per tier, only the *search index* gets built per tier later, once the scaling evaluation notebook exists.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULT_DIR = Path("result/44_build_scaled_corpus")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

existing = pd.read_csv("dataset/company_corpus.csv").drop_duplicates(subset="domain").reset_index(drop=True)
new_sample = pd.read_parquet("dataset/goi_random_sample_300k_clean.parquet")

# Drop the ~0.56% of the random sample that's already in the existing corpus -- avoid double-counting.
new_only = new_sample[~new_sample["already_in_existing_corpus"]].reset_index(drop=True)

print(f"Existing corpus       : {len(existing):,} companies (all gold-labelled data lives here)")
print(f"New random sample     : {len(new_sample):,} companies")
print(f"New, non-overlapping  : {len(new_only):,} companies")
print(f"Combined pool (max)   : {len(existing) + len(new_only):,} companies")

Existing corpus       : 98,716 companies (all gold-labelled data lives here)
New random sample     : 300,000 companies
New, non-overlapping  : 298,309 companies
Combined pool (max)   : 397,025 companies


In [2]:
def build_rich_text(row):
    """Combine all informative fields into one string -- same convention as 03_baseline_minilm.ipynb / 20_baseline_linq_mistral.ipynb."""
    parts = []
    for field, prefix in [
        ("name",              "Company:"),
        ("country",           "Country:"),
        ("state",             "State:"),
        ("municipality",      "City:"),
        ("district",          "District:"),
        ("organization_type", "Type:"),
        ("organization_size", "Size:"),
        ("nace_code",         "Industry:"),
        ("summary",           ""),
    ]:
        val = row.get(field, "")
        if isinstance(val, str) and val.strip():
            parts.append(f"{prefix} {val}".strip() if prefix else val)
    kw = row.get("summary_keywords", "")
    if isinstance(kw, str) and kw.strip():
        kw_clean = kw.replace("'", "").replace("[", "").replace("]", "")
        parts.append(f"Keywords: {kw_clean}")
    return " | ".join(parts)


SHARED_COLS = ["domain", "name", "organization_type", "organization_size", "country",
               "state", "district", "municipality", "summary", "summary_keywords", "nace_code"]

existing_slim = existing[SHARED_COLS].copy()
existing_slim["source"] = "existing"
new_slim = new_only[SHARED_COLS].copy()
new_slim["source"] = "new"

combined = pd.concat([existing_slim, new_slim], ignore_index=True)
combined = combined.drop_duplicates(subset="domain").reset_index(drop=True)

print("[Text] Building rich text for every company in the combined pool...")
combined["rich_text"] = [build_rich_text(row) for _, row in combined.iterrows()]
print(f"[Text] Done. Combined pool: {len(combined):,} rows")
print(f"[Text] Sample: {combined['rich_text'].iloc[0][:200]}...")

[Text] Building rich text for every company in the combined pool...
[Text] Done. Combined pool: 397,025 rows
[Text] Sample: Company: Software Genesis, Inc. | Country: United States | State: Illinois | Type: Company | Size: Micro (0-9) | Industry: NACE K: Telecommunication, computer programming, consulting, computing infras...


In [3]:
RANDOM_SEED = 42
TARGET_SIZES = {"100k": 100_000, "200k": 200_000, "300k": 300_000}
# The 4th, largest tier is just "everything available" -- existing + all non-overlapping new
# companies -- rather than an exact 400,000, since that's however many actually exist.

new_indices = combined.index[combined["source"] == "new"].to_numpy()
rng = np.random.default_rng(RANDOM_SEED)
shuffled_new = rng.permutation(new_indices)  # one fixed shuffle -- every tier is a prefix of this

existing_mask = (combined["source"] == "existing").to_numpy()
n_existing = existing_mask.sum()

for tier_name, target_size in TARGET_SIZES.items():
    n_new_needed = max(0, target_size - n_existing)
    n_new_needed = min(n_new_needed, len(shuffled_new))
    included_new = set(shuffled_new[:n_new_needed])
    combined[f"in_{tier_name}"] = existing_mask | combined.index.isin(included_new)
    print(f"Tier {tier_name:>5} (target {target_size:>7,}): {combined[f'in_{tier_name}'].sum():>7,} companies "
          f"({n_existing:,} existing + {n_new_needed:,} new)")

combined["in_400k"] = True  # everything -- existing + all available new
print(f"Tier  400k (target  ~400,000): {combined['in_400k'].sum():>7,} companies "
      f"({n_existing:,} existing + {len(new_indices):,} new)")

Tier  100k (target 100,000): 100,000 companies (98,716 existing + 1,284 new)
Tier  200k (target 200,000): 200,000 companies (98,716 existing + 101,284 new)
Tier  300k (target 300,000): 300,000 companies (98,716 existing + 201,284 new)
Tier  400k (target  ~400,000): 397,025 companies (98,716 existing + 298,309 new)


In [4]:
out_path = RESULT_DIR / "combined_pool.parquet"
combined.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}")
print(f"Columns: {list(combined.columns)}")
print()
print("This file is the shared input for notebooks 45 (MiniLM), 46 (GTE-large), and 47 (Linq-Embed-Mistral).")
print("Each encodes combined['rich_text'] once, in the same row order, so embeddings[i] always corresponds to combined.iloc[i].")

Saved -> result/44_build_scaled_corpus/combined_pool.parquet
Columns: ['domain', 'name', 'organization_type', 'organization_size', 'country', 'state', 'district', 'municipality', 'summary', 'summary_keywords', 'nace_code', 'source', 'rich_text', 'in_100k', 'in_200k', 'in_300k', 'in_400k']

This file is the shared input for notebooks 45 (MiniLM), 46 (GTE-large), and 47 (Linq-Embed-Mistral).
Each encodes combined['rich_text'] once, in the same row order, so embeddings[i] always corresponds to combined.iloc[i].
